In [1]:
SEED = 42
import numpy as np
import torch
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
from pathlib import Path
Path("../reports/figures").mkdir(parents=True, exist_ok=True)
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import matplotlib.pyplot as plt

DATA_DIR   = Path("../data/processed/FD001")
MODELS_DIR = Path("../data/processed/FD001/checkpoints_cgan")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X      = np.load(DATA_DIR / "X_train.npy")
labels = np.load(DATA_DIR / "labels_train.npy")

INPUT_DIM   = X.shape[2]
SEQ_LEN     = X.shape[1]
LATENT_DIM  = 64
HIDDEN_DIM  = 128
NUM_CLASSES = 4
EMBED_DIM   = 16
BATCH_SIZE  = 64

cfg = {
    "seq_len"    : SEQ_LEN,
    "input_dim"  : INPUT_DIM,
    "hidden_dim" : HIDDEN_DIM,
    "latent_dim" : LATENT_DIM,
    "num_classes": NUM_CLASSES,
    "embed_dim"  : EMBED_DIM,
    "model_type" : "cgan",
}
with open(MODELS_DIR / "model_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print(f"X      : {X.shape}")
print(f"Device : {DEVICE}")
print(f"Config : {cfg}")

X      : (17731, 30, 17)
Device : cpu
Config : {'seq_len': 30, 'input_dim': 17, 'hidden_dim': 128, 'latent_dim': 64, 'num_classes': 4, 'embed_dim': 16, 'model_type': 'cgan'}


In [ ]:
# ── BUG-013: Early stopping helper ──────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler as SKStdScaler

EARLY_STOP_THRESHOLD = 0.60
EVAL_INTERVAL = 50


def compute_discriminative_score(G_model, X_real, y_real, n_samples=500, seed=42):
    """
    Draw n_samples real and n_samples synthetic windows.
    Train a RandomForest in 5-fold CV to distinguish them.
    Returns mean accuracy (ideal ~0.50).
    """
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(X_real), size=n_samples, replace=False)
    X_real_samp = X_real[indices].reshape(n_samples, -1)
    y_real_cls = y_real[indices]

    with torch.no_grad():
        z = torch.randn(n_samples, LATENT_DIM).to(DEVICE)
        c = torch.tensor(y_real_cls, dtype=torch.long).to(DEVICE)
        X_fake_t = G_model(z, c).cpu().numpy()
    X_fake_samp = X_fake_t.reshape(n_samples, -1)

    X_all = np.vstack([X_real_samp, X_fake_samp])
    y_disc = np.array([0] * n_samples + [1] * n_samples)

    sc = SKStdScaler()
    X_all = sc.fit_transform(X_all)

    clf = RandomForestClassifier(n_estimators=100, random_state=seed)
    scores = cross_val_score(clf, X_all, y_disc, cv=5)
    return scores.mean()


In [2]:
class CMAPSSDataset(Dataset):
    def __init__(self, X, labels):
        self.X      = torch.tensor(X,      dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.labels[idx]


dataset    = CMAPSSDataset(X, labels)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"Dataset size  : {len(dataset)}")
print(f"Batches/epoch : {len(dataloader)}")

Dataset size  : 17731
Batches/epoch : 277


In [3]:
class CGANGenerator(nn.Module):
    """
    Noise Z (latent_dim) + class embedding -> sensor sequence (seq_len, input_dim)
    Operates directly in sensor space — no latent GRU encoding.
    """
    def __init__(self, latent_dim, hidden_dim, seq_len,
                 output_dim, num_classes, embed_dim):
        super().__init__()
        self.seq_len = seq_len

        self.class_embed = nn.Embedding(num_classes, embed_dim)

        # project noise + embedding -> hidden
        self.fc_in = nn.Sequential(
            nn.Linear(latent_dim + embed_dim, hidden_dim),
            nn.LeakyReLU(0.2),
        )

        # temporal GRU to generate sequence
        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
        )

        # project hidden -> sensor values
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim // 2, output_dim),
            nn.Sigmoid(),   # output in [0,1]
        )

    def forward(self, z, c):
        # z: (B, latent_dim)   c: (B,)
        emb = self.class_embed(c)                      # (B, embed_dim)
        inp = torch.cat([z, emb], dim=-1)              # (B, latent+embed)
        h   = self.fc_in(inp)                          # (B, hidden)

        # repeat across time
        h_seq = h.unsqueeze(1).repeat(1, self.seq_len, 1)  # (B, T, hidden)

        # add noise at every timestep to prevent collapse
        h_seq = h_seq + 0.1 * torch.randn_like(h_seq)

        out, _ = self.gru(h_seq)                       # (B, T, hidden)
        return self.fc_out(out)                        # (B, T, input_dim)


G = CGANGenerator(LATENT_DIM, HIDDEN_DIM, SEQ_LEN,
                  INPUT_DIM, NUM_CLASSES, EMBED_DIM).to(DEVICE)
print(G)
print(f"\nGenerator params: {sum(p.numel() for p in G.parameters()):,}")

CGANGenerator(
  (class_embed): Embedding(4, 16)
  (fc_in): Sequential(
    (0): Linear(in_features=80, out_features=128, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
  )
  (gru): GRU(128, 128, num_layers=2, batch_first=True)
  (fc_out): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=64, out_features=17, bias=True)
    (3): Sigmoid()
  )
)

Generator params: 217,937


In [4]:
class CGANDiscriminator(nn.Module):
    """
    Sensor sequence (seq_len, input_dim) + class embedding -> real/fake score
    Uses GRU to process temporal sequence.
    """
    def __init__(self, input_dim, hidden_dim,
                 num_classes, embed_dim):
        super().__init__()
        self.class_embed = nn.Embedding(num_classes, embed_dim)

        self.gru = nn.GRU(
            input_size=input_dim + embed_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
        )

        self.fc_out = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x, c):
        # x: (B, T, input_dim)   c: (B,)
        emb   = self.class_embed(c).unsqueeze(1).repeat(1, x.size(1), 1)
        inp   = torch.cat([x, emb], dim=-1)            # (B, T, input+embed)
        _, h  = self.gru(inp)                          # h: (2, B, hidden)
        h_last = h[-1]                                 # (B, hidden) last layer
        return self.fc_out(h_last)                     # (B, 1)


D = CGANDiscriminator(INPUT_DIM, HIDDEN_DIM,
                      NUM_CLASSES, EMBED_DIM).to(DEVICE)
print(D)
print(f"\nDiscriminator params: {sum(p.numel() for p in D.parameters()):,}")

CGANDiscriminator(
  (class_embed): Embedding(4, 16)
  (gru): GRU(33, 128, num_layers=2, batch_first=True)
  (fc_out): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

Discriminator params: 170,049


In [5]:
with torch.no_grad():
    z_test = torch.randn(4, LATENT_DIM).to(DEVICE)
    c_test = torch.randint(0, NUM_CLASSES, (4,)).to(DEVICE)
    x_fake = G(z_test, c_test)
    score  = D(x_fake, c_test)

print(f"Generator output : {x_fake.shape}")   # (4, 30, 17)
print(f"Discriminator out: {score.shape}")     # (4, 1)
print(f"Output range     : [{x_fake.min():.3f}, {x_fake.max():.3f}]")
print("Smoke test passed.")

Generator output : torch.Size([4, 30, 17])
Discriminator out: torch.Size([4, 1])
Output range     : [0.459, 0.546]
Smoke test passed.


In [ ]:
LR_G   = 1e-4
LR_D   = 2e-4
EPOCHS = 500

opt_G = torch.optim.Adam(G.parameters(), lr=LR_G, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.999))
bce   = nn.BCEWithLogitsLoss()

from src.utils.cgan_training import correlation_penalty, EarlyStopping
stopper = EarlyStopping(
    G=G, D=D, X_real=X, y_real=labels,
    latent_dim=LATENT_DIM, device=DEVICE,
    ckpt_dir=MODELS_DIR,
    eval_interval=50, patience=3
)
history_g, history_d = [], []

print(f"Training CGAN for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    g_epoch, d_epoch = 0.0, 0.0

    for x_real, c_batch in dataloader:
        x_real  = x_real.to(DEVICE)
        c_batch = c_batch.to(DEVICE)
        B       = x_real.size(0)
        z       = torch.randn(B, LATENT_DIM).to(DEVICE)

        # ── Discriminator ────────────────────────────────────────────
        x_fake  = G(z, c_batch).detach()
        d_real  = D(x_real, c_batch)
        d_fake  = D(x_fake, c_batch)

        loss_D  = (bce(d_real, torch.ones_like(d_real)) +
                   bce(d_fake, torch.zeros_like(d_fake))) / 2.0

        opt_D.zero_grad()
        loss_D.backward()
        torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
        opt_D.step()

        # ── Generator (train 2x per D step) ──────────────────────────
        for _ in range(2):
            z      = torch.randn(B, LATENT_DIM).to(DEVICE)
            x_fake = G(z, c_batch)
            d_fake = D(x_fake, c_batch)

            loss_G = bce(d_fake, torch.ones_like(d_fake))

            # feature matching loss — match real/fake means per feature
            loss_fm = torch.mean(torch.abs(
                x_real.mean(dim=(0,1)) - x_fake.mean(dim=(0,1))
            ))

            loss_corr = correlation_penalty(x_real, x_fake)
            loss_G_total = loss_G + 10.0 * loss_fm + 0.5 * loss_corr

            opt_G.zero_grad()
            loss_G_total.backward()
            torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
            opt_G.step()

        g_epoch += loss_G_total.item()
        d_epoch += loss_D.item()

    g_avg = g_epoch / len(dataloader)
    d_avg = d_epoch / len(dataloader)
    history_g.append(g_avg)
    history_d.append(d_avg)

    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1:>4}/{EPOCHS}  G={g_avg:.4f}  D={d_avg:.4f}")

    if stopper.step(epoch):
        print("Early stopping triggered.")
        break

stopper.restore_best()
print("Training complete.")

Training CGAN for 500 epochs...
  Epoch    1/500  G=0.9887  D=0.6449
  Epoch    2/500  G=1.0011  D=0.6345


KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(history_g, label="Generator")
ax.plot(history_d, label="Discriminator")
ax.set_title("CGAN Training Losses")
ax.set_xlabel("Epoch")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/cgan_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Final G loss: {history_g[-1]:.4f}")
print(f"Final D loss: {history_d[-1]:.4f}")
print("Healthy training: D loss should be ~0.5-0.7, G loss ~0.5-2.0")

In [ ]:
torch.save(G.state_dict(), MODELS_DIR / "generator.pt")
torch.save(D.state_dict(), MODELS_DIR / "discriminator.pt")
print(f"Checkpoints saved -> {MODELS_DIR.resolve()}")

In [ ]:
CLASS_NAMES  = ["C0 healthy", "C1 early", "C2 advanced", "C3 imminent"]
FEATURE_COLS = [
    "op1","op2","op3",
    "s2","s3","s4","s7","s8","s9",
    "s11","s12","s13","s14","s15","s17","s20","s21",
]

G.eval()
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

with torch.no_grad():
    for c in range(4):
        z        = torch.randn(1, LATENT_DIM).to(DEVICE)
        c_tensor = torch.tensor([c]).to(DEVICE)
        x_hat    = G(z, c_tensor).cpu().numpy()[0]   # (30, 17)

        ax = axes[c]
        for i, col in enumerate(FEATURE_COLS[:6]):
            ax.plot(x_hat[:, i], label=col, alpha=0.8)
        ax.set_title(f"Synthetic — {CLASS_NAMES[c]}")
        ax.set_ylabel("Norm. value")
        ax.legend(loc="upper right", fontsize=7)

axes[-1].set_xlabel("Timestep")
plt.tight_layout()
plt.savefig("../reports/figures/cgan_synthetic_samples.png", dpi=150, bbox_inches="tight")
plt.show()